# Notebook 02b — OCR for Poor-Quality PDFs

This notebook processes only documents that failed normal text extraction.

Pipeline:
1. Load extraction register
2. Build OCR queue
3. Convert PDF pages → images (Poppler)
4. OCR images → text (Tesseract)
5. Save OCR text + update registers

In [42]:
from pathlib import Path
from datetime import datetime, timezone
import os

import pandas as pd
import pytesseract
from pdf2image import convert_from_path

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
META_DIR = DATA_DIR / "metadata"
PROCESSED_DIR = DATA_DIR / "processed"

OCR_DIR = PROCESSED_DIR / "ocr"
OCR_TEXT_DIR = OCR_DIR / "text"
OCR_IMAGE_DIR = OCR_DIR / "pages"

for d in [OCR_TEXT_DIR, OCR_IMAGE_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# External tools
POPPLER_PATH = r"C:\Users\Administrator\Downloads\Release-24.08.0-0\poppler-24.08.0\Library\bin"
TESSDATA_DIR = r"C:\tessdata"

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
os.environ["TESSDATA_PREFIX"] = TESSDATA_DIR
OCR_CONFIG = f"--tessdata-dir {TESSDATA_DIR}"

# Registers
register_path = META_DIR / "exam_document_register.csv"
extraction_register_path = META_DIR / "document_extraction_register.csv"

register_df = pd.read_csv(register_path)
extraction_df = pd.read_csv(extraction_register_path)

print("Setup complete")
print("eng.traineddata:", Path(TESSDATA_DIR, "eng.traineddata").exists())
print("pdftoppm.exe   :", (Path(POPPLER_PATH) / "pdftoppm.exe").exists())

Setup complete
eng.traineddata: True
pdftoppm.exe   : True


In [43]:
ocr_queue = extraction_df[
    (extraction_df["extraction_status"] == "extracted")
    & (extraction_df["ocr_required"] == True)
].copy()

print(f"Documents requiring OCR: {len(ocr_queue)}")

if len(ocr_queue) == 0:
    print("No OCR required. Notebook 02b complete.")
else:
    display(
        ocr_queue[
            [
                "document_id",
                "file_name",
                "page_count",
                "character_count",
                "extraction_quality",
            ]
        ]
    )

Documents requiring OCR: 7


,document_id,file_name,page_count,character_count,extraction_quality
0,2023_nov_p1_exam_maths,NSC_Mathematics_2023_Nov_P1_exam.pdf,10,0,poor
2,2023_nov_p2_exam_maths,NSC_Mathematics_2023_Nov_P2_exam.pdf,14,0,poor
4,2024_nov_p1_exam_maths,NSC_Mathematics_2024_Nov_P1_exam.pdf,10,0,poor
6,2024_nov_p2_exam_maths,NSC_Mathematics_2024_Nov_P2_exam.pdf,14,0,poor
8,2025_nov_p1_exam_maths,NSC_Mathematics_2025_Nov_P1_exam.pdf,12,0,poor
10,2025_nov_p2_exam_maths,NSC_Mathematics_2025_Nov_P2_exam.pdf,15,0,poor
11,2025_nov_p2_memo_maths,NSC_Mathematics_2025_Nov_P2_memo.pdf,12,0,poor


In [44]:
def ocr_pdf(pdf_path: Path, document_id: str, dpi: int = 200):
    """Convert PDF pages to images and OCR each page."""
    page_images = convert_from_path(
        str(pdf_path),
        dpi=dpi,
        poppler_path=POPPLER_PATH,
    )

    page_texts = []
    doc_image_dir = OCR_IMAGE_DIR / document_id
    doc_image_dir.mkdir(parents=True, exist_ok=True)

    for i, image in enumerate(page_images, start=1):
        image_path = doc_image_dir / f"page_{i:02d}.png"
        image.save(image_path, "PNG")

        text = pytesseract.image_to_string(
            image,
            lang="eng",
            config=OCR_CONFIG,
        )
        page_texts.append(text.strip())

    full_text = "\n\n".join(page_texts)
    return full_text, len(page_images)


print("OCR helper ready")

OCR helper ready


In [45]:
ocr_results = []

for _, row in ocr_queue.iterrows():
    document_id = row["document_id"]
    pdf_path = PROJECT_ROOT / row["pdf_path"]

    print(f"\nOCR: {document_id}")

    if not pdf_path.exists():
        ocr_results.append(
            {
                "document_id": document_id,
                "ocr_status": "failed",
                "reason": "PDF not found",
            }
        )
        continue

    try:
        full_text, page_count = ocr_pdf(pdf_path, document_id)
        text_path = OCR_TEXT_DIR / f"{document_id}_ocr.txt"
        text_path.write_text(full_text, encoding="utf-8")

        print(f"  pages={page_count} | chars={len(full_text)}")

        ocr_results.append(
            {
                "document_id": document_id,
                "ocr_text_file": str(text_path.relative_to(PROJECT_ROOT)),
                "page_count": page_count,
                "character_count": len(full_text),
                "ocr_status": "ocr_complete",
                "ocr_date": datetime.now(timezone.utc).isoformat(),
            }
        )

    except Exception as e:
        print(f"  ERROR: {e}")
        ocr_results.append(
            {
                "document_id": document_id,
                "ocr_status": "failed",
                "reason": str(e),
            }
        )

ocr_df = pd.DataFrame(ocr_results)
display(ocr_df)


OCR: 2023_nov_p1_exam_maths
  ERROR: (1, 'Error opening data file C:\\tessdata/eng.traineddata Please make sure the TESSDATA_PREFIX environment variable is set to your "tessdata" directory. Failed loading language \'eng\' Tesseract couldn\'t load any languages! Could not initialize tesseract.')

OCR: 2023_nov_p2_exam_maths
  ERROR: (1, 'Error opening data file C:\\tessdata/eng.traineddata Please make sure the TESSDATA_PREFIX environment variable is set to your "tessdata" directory. Failed loading language \'eng\' Tesseract couldn\'t load any languages! Could not initialize tesseract.')

OCR: 2024_nov_p1_exam_maths
  ERROR: (1, 'Error opening data file C:\\tessdata/eng.traineddata Please make sure the TESSDATA_PREFIX environment variable is set to your "tessdata" directory. Failed loading language \'eng\' Tesseract couldn\'t load any languages! Could not initialize tesseract.')

OCR: 2024_nov_p2_exam_maths
  ERROR: (1, 'Error opening data file C:\\tessdata/eng.traineddata Please make s

,document_id,ocr_status,reason
0,2023_nov_p1_exam_maths,failed,"(1, 'Error opening data file C:\\tessdata/eng...."
1,2023_nov_p2_exam_maths,failed,"(1, 'Error opening data file C:\\tessdata/eng...."
2,2024_nov_p1_exam_maths,failed,"(1, 'Error opening data file C:\\tessdata/eng...."
3,2024_nov_p2_exam_maths,failed,"(1, 'Error opening data file C:\\tessdata/eng...."
4,2025_nov_p1_exam_maths,failed,"(1, 'Error opening data file C:\\tessdata/eng...."
5,2025_nov_p2_exam_maths,failed,"(1, 'Error opening data file C:\\tessdata/eng...."
6,2025_nov_p2_memo_maths,failed,"(1, 'Error opening data file C:\\tessdata/eng...."


In [1]:
from pathlib import Path

eng = Path(r"C:\tessdata\eng.traineddata")
print("exists:", eng.exists())
print("size bytes:", eng.stat().st_size if eng.exists() else None)

exists: True
size bytes: 0


In [2]:
import urllib.request
from pathlib import Path

dest = Path(r"C:\tessdata\eng.traineddata")
dest.parent.mkdir(parents=True, exist_ok=True)

urls = [
    "https://cdn.jsdelivr.net/gh/tesseract-ocr/tessdata@main/eng.traineddata",
    "https://github.com/tesseract-ocr/tessdata/raw/main/eng.traineddata",
]

for url in urls:
    try:
        print("Downloading from:", url)
        urllib.request.urlretrieve(url, dest)
        size = dest.stat().st_size
        print("Saved size:", size)
        if size > 1_000_000:  # real file should be multi-MB
            print("SUCCESS")
            break
        else:
            print("File too small, trying next source...")
    except Exception as e:
        print("Failed:", e)

Failed: HTTP Error 403: Forbidden
Failed: [Errno 13] Permission denied: 'C:\\tessdata\\eng.traineddata'


In [3]:
import urllib.request
from pathlib import Path

dest = Path(r"C:\tessdata\eng.traineddata")
dest.parent.mkdir(parents=True, exist_ok=True)

url = "https://cdn.jsdelivr.net/gh/tesseract-ocr/tessdata_fast@main/eng.traineddata"

print("Downloading fast eng model...")
urllib.request.urlretrieve(url, dest)
print("size bytes:", dest.stat().st_size)

PermissionError: [Errno 13] Permission denied: 'C:\\tessdata\\eng.traineddata'

In [4]:
import os
from pathlib import Path
import pytesseract

TESSDATA_DIR = str(Path.home() / "tessdata")
POPPLER_PATH = r"C:\Users\Administrator\Downloads\Release-24.08.0-0\poppler-24.08.0\Library\bin"

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
os.environ["TESSDATA_PREFIX"] = TESSDATA_DIR
OCR_CONFIG = f"--tessdata-dir {TESSDATA_DIR}"

print("tessdata dir:", TESSDATA_DIR)
print("eng exists:", (Path(TESSDATA_DIR) / "eng.traineddata").exists())
print("eng is file:", (Path(TESSDATA_DIR) / "eng.traineddata").is_file())
print("size:", (Path(TESSDATA_DIR) / "eng.traineddata").stat().st_size)


tessdata dir: C:\Users\Administrator\tessdata
eng exists: True
eng is file: True
size: 4113088


In [5]:
import os
from pathlib import Path
import pytesseract
from pdf2image import convert_from_path

TESSDATA_DIR = str(Path.home() / "tessdata")
POPPLER_PATH = r"C:\Users\Administrator\Downloads\Release-24.08.0-0\poppler-24.08.0\Library\bin"

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
os.environ["TESSDATA_PREFIX"] = TESSDATA_DIR
OCR_CONFIG = f"--tessdata-dir {TESSDATA_DIR}"

print("languages:", pytesseract.get_languages(config=OCR_CONFIG))

languages: []


In [6]:
test_pdf = PROJECT_ROOT / ocr_queue.iloc[0]["pdf_path"]

pages = convert_from_path(
    str(test_pdf),
    dpi=150,
    first_page=1,
    last_page=1,
    poppler_path=POPPLER_PATH
)

text = pytesseract.image_to_string(
    pages[0],
    lang="eng",
    config=OCR_CONFIG
)

print(text[:1000])

NameError: name 'PROJECT_ROOT' is not defined

In [7]:
from pathlib import Path
from datetime import datetime, timezone
import os

import pandas as pd
import pytesseract
from pdf2image import convert_from_path

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
META_DIR = DATA_DIR / "metadata"
PROCESSED_DIR = DATA_DIR / "processed"

OCR_DIR = PROCESSED_DIR / "ocr"
OCR_TEXT_DIR = OCR_DIR / "text"
OCR_IMAGE_DIR = OCR_DIR / "pages"
OCR_TEXT_DIR.mkdir(parents=True, exist_ok=True)
OCR_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# Tools
TESSDATA_DIR = str(Path.home() / "tessdata")
POPPLER_PATH = r"C:\Users\Administrator\Downloads\Release-24.08.0-0\poppler-24.08.0\Library\bin"

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
os.environ["TESSDATA_PREFIX"] = TESSDATA_DIR
OCR_CONFIG = f"--tessdata-dir {TESSDATA_DIR}"

# Registers
register_path = META_DIR / "exam_document_register.csv"
extraction_register_path = META_DIR / "document_extraction_register.csv"

register_df = pd.read_csv(register_path)
extraction_df = pd.read_csv(extraction_register_path)

# OCR queue
ocr_queue = extraction_df[
    (extraction_df["extraction_status"] == "extracted")
    & (extraction_df["ocr_required"] == True)
].copy()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OCR queue size:", len(ocr_queue))
print("eng size:", (Path(TESSDATA_DIR) / "eng.traineddata").stat().st_size)

PROJECT_ROOT: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence
OCR queue size: 7
eng size: 4113088


In [8]:
test_pdf = PROJECT_ROOT / ocr_queue.iloc[0]["pdf_path"]

pages = convert_from_path(
    str(test_pdf),
    dpi=150,
    first_page=1,
    last_page=1,
    poppler_path=POPPLER_PATH
)

text = pytesseract.image_to_string(
    pages[0],
    lang="eng",
    config=OCR_CONFIG
)

print(text[:1000])

basic education

Department:
Basic Education
REPUBLIC OF SOUTH AFRICA

NATIONAL
SENIOR CERTIFICATE

GRADE 12

MATHEMATICS P1

NOVEMBER 2023

TIME: 3 hours

This question paper consists of 9 pages and 1 information sheet.

Copyright reserved Please turn over




In [9]:
def ocr_pdf(pdf_path: Path, document_id: str, dpi: int = 200):
    page_images = convert_from_path(
        str(pdf_path),
        dpi=dpi,
        poppler_path=POPPLER_PATH
    )

    page_texts = []
    doc_image_dir = OCR_IMAGE_DIR / document_id
    doc_image_dir.mkdir(parents=True, exist_ok=True)

    for i, image in enumerate(page_images, start=1):
        image_path = doc_image_dir / f"page_{i:02d}.png"
        image.save(image_path, "PNG")
        text = pytesseract.image_to_string(
            image,
            lang="eng",
            config=OCR_CONFIG
        )
        page_texts.append(text.strip())

    return "\n\n".join(page_texts), len(page_images)

print("Helper ready")

Helper ready


In [10]:
ocr_results = []

for _, row in ocr_queue.iterrows():
    document_id = row["document_id"]
    pdf_path = PROJECT_ROOT / row["pdf_path"]
    print(f"\nOCR: {document_id}")

    if not pdf_path.exists():
        ocr_results.append({
            "document_id": document_id,
            "ocr_status": "failed",
            "reason": "PDF not found",
        })
        continue

    try:
        full_text, page_count = ocr_pdf(pdf_path, document_id)
        text_path = OCR_TEXT_DIR / f"{document_id}_ocr.txt"
        text_path.write_text(full_text, encoding="utf-8")
        print(f"  pages={page_count} | chars={len(full_text)}")

        ocr_results.append({
            "document_id": document_id,
            "ocr_text_file": str(text_path.relative_to(PROJECT_ROOT)),
            "page_count": page_count,
            "character_count": len(full_text),
            "ocr_status": "ocr_complete",
            "ocr_date": datetime.now(timezone.utc).isoformat(),
        })
    except Exception as e:
        print(f"  ERROR: {e}")
        ocr_results.append({
            "document_id": document_id,
            "ocr_status": "failed",
            "reason": str(e),
        })

ocr_df = pd.DataFrame(ocr_results)
display(ocr_df)


OCR: 2023_nov_p1_exam_maths
  pages=10 | chars=8423

OCR: 2023_nov_p2_exam_maths
  pages=14 | chars=9620

OCR: 2024_nov_p1_exam_maths
  pages=10 | chars=9737

OCR: 2024_nov_p2_exam_maths
  pages=14 | chars=10389

OCR: 2025_nov_p1_exam_maths
  pages=12 | chars=8995

OCR: 2025_nov_p2_exam_maths
  pages=15 | chars=9548

OCR: 2025_nov_p2_memo_maths
  pages=12 | chars=8995


,document_id,ocr_text_file,page_count,character_count,ocr_status,ocr_date
0,2023_nov_p1_exam_maths,data\processed\ocr\text\2023_nov_p1_exam_maths...,10,8423,ocr_complete,2026-09-10T07:29:31.208230+00:00
1,2023_nov_p2_exam_maths,data\processed\ocr\text\2023_nov_p2_exam_maths...,14,9620,ocr_complete,2026-09-10T07:30:07.335497+00:00
2,2024_nov_p1_exam_maths,data\processed\ocr\text\2024_nov_p1_exam_maths...,10,9737,ocr_complete,2026-09-10T07:30:28.254302+00:00
3,2024_nov_p2_exam_maths,data\processed\ocr\text\2024_nov_p2_exam_maths...,14,10389,ocr_complete,2026-09-10T07:30:55.468737+00:00
4,2025_nov_p1_exam_maths,data\processed\ocr\text\2025_nov_p1_exam_maths...,12,8995,ocr_complete,2026-09-10T07:31:17.439608+00:00
5,2025_nov_p2_exam_maths,data\processed\ocr\text\2025_nov_p2_exam_maths...,15,9548,ocr_complete,2026-09-10T07:31:44.562548+00:00
6,2025_nov_p2_memo_maths,data\processed\ocr\text\2025_nov_p2_memo_maths...,12,8995,ocr_complete,2026-09-10T07:32:06.635866+00:00


In [11]:
ocr_register_path = META_DIR / "document_ocr_register.csv"
ocr_df.to_csv(ocr_register_path, index=False)

for _, row in ocr_df.iterrows():
    if row.get("ocr_status") != "ocr_complete":
        continue

    idx = extraction_df.index[extraction_df["document_id"] == row["document_id"]]
    if len(idx):
        extraction_df.loc[idx, "ocr_required"] = False
        extraction_df.loc[idx, "extraction_quality"] = "ocr_good"
        extraction_df.loc[idx, "character_count"] = row.get("character_count", 0)
        extraction_df.loc[idx, "text_file"] = row.get("ocr_text_file")

    idx = register_df.index[register_df["document_id"] == row["document_id"]]
    if len(idx):
        register_df.loc[idx, "ocr_required"] = False
        register_df.loc[idx, "text_extractable"] = True
        register_df.loc[idx, "extraction_quality"] = "ocr_good"

extraction_df.to_csv(extraction_register_path, index=False)
register_df.to_csv(register_path, index=False)

print("Saved:", ocr_register_path)
print("Registers updated")

Saved: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\document_ocr_register.csv
Registers updated


In [12]:
complete = int((ocr_df["ocr_status"] == "ocr_complete").sum()) if len(ocr_df) else 0
failed = int((ocr_df["ocr_status"] == "failed").sum()) if len(ocr_df) else 0

print("=" * 60)
print("NOTEBOOK 02b COMPLETE")
print("=" * 60)
print(f"OCR queue     : {len(ocr_queue)}")
print(f"OCR complete  : {complete}")
print(f"OCR failed    : {failed}")
print(f"OCR text dir  : {OCR_TEXT_DIR}")
print("=" * 60)

NOTEBOOK 02b COMPLETE
OCR queue     : 7
OCR complete  : 7
OCR failed    : 0
OCR text dir  : c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\ocr\text
